In [1]:
import spacy
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [2]:
nlp = spacy.load("en_core_web_sm")

In [3]:
reviews = [
    # POSITIVE (1)
    "nice food and quick service",
    "amazing restaurant, loved the ambience",
    "too good, would recommend",
    "just loved it, great taste",
    "will go again for sure",
    "food was delicious and fresh",
    "staff was polite and helpful",
    "excellent service and clean place",
    "great value for money",
    "best burger i have had here",
    "the pasta was perfect and creamy",
    "superb experience, everything was on point",
    "really enjoyed the meal",
    "fast delivery and tasty food",
    "the dessert was amazing",
    "portion size was good and filling",
    "the place is cozy and comfortable",
    "highly recommended, loved it",
    "good food, good vibes",
    "the spices were balanced and flavorful",
    "the chicken was juicy and well cooked",
    "service was prompt and friendly",
    "fresh ingredients and great taste",
    "loved the presentation and quality",
    "worth visiting again",
    # NEGATIVE (0)
    "horrible food and bad smell",
    "never go there, waste of money",
    "poor service and rude staff",
    "poor quality food, not fresh",
    "needs improvement, very disappointing",
    "food was cold and tasteless",
    "overpriced and not worth it",
    "worst experience ever",
    "staff was arrogant and unhelpful",
    "waited too long for the order",
    "the rice was undercooked",
    "the chicken was raw inside",
    "stale bread and oily fries",
    "the place was dirty and messy",
    "they messed up my order",
    "customer service was terrible",
    "not recommended at all",
    "i got sick after eating here",
    "flavor was bland and boring",
    "the sauce tasted weird",
    "portion was too small for the price",
    "delivery was late and food spilled",
    "very bad hygiene",
    "waste of time and money",
    "i will never come back",
    # HARD / REALISTIC MIXED CASES (label them based on overall tone)
    "food was good but service was slow",  # 1 (overall okay)
    "taste was nice but portion was small",  # 1
    "ambience was great but food was average",  # 0
    "not bad, but nothing special",  # 0
    "expected better for the price",  # 0
    "okay food, okay service",  # 0
    "good service but the food was cold",  # 0
    "the food was decent, might try again",  # 1
    "it was fine, not amazing",  # 0
    "great taste but too expensive",  # 1
]

sentiment = np.array(
    [
        # 25 positive
        *([1] * 25),
        # 25 negative
        *([0] * 25),
        # 10 mixed
        1,
        1,
        0,
        0,
        0,
        0,
        0,
        1,
        0,
        1,
    ]
)


In [4]:
from tensorflow.keras.preprocessing.text import one_hot

print(one_hot(reviews[0], 50))

[5, 20, 2, 18, 10]


In [5]:
VOCAB_LENGTH = 50
encoded_reviews = [one_hot(txt, VOCAB_LENGTH) for txt in reviews]
encoded_reviews

[[5, 20, 2, 18, 10],
 [27, 18, 24, 43, 43],
 [18, 48, 4, 41],
 [45, 24, 2, 14, 22],
 [41, 11, 37, 26, 12],
 [20, 39, 36, 2, 14],
 [24, 39, 15, 2, 42],
 [10, 10, 2, 46, 42],
 [14, 22, 26, 42],
 [41, 32, 15, 11, 37, 46],
 [43, 5, 39, 37, 2, 24],
 [25, 22, 2, 39, 25, 31],
 [8, 49, 43, 27],
 [9, 23, 2, 47, 20],
 [43, 23, 39, 27],
 [37, 11, 39, 48, 2, 36],
 [43, 42, 35, 41, 2, 17],
 [48, 30, 24, 2],
 [48, 20, 48, 39],
 [43, 49, 47, 33, 2, 40],
 [43, 27, 39, 17, 2, 29, 5],
 [10, 39, 47, 2, 42],
 [14, 20, 2, 14, 22],
 [24, 43, 38, 2, 5],
 [44, 2, 37],
 [42, 20, 2, 32, 3],
 [18, 11, 35, 16, 42, 42],
 [24, 10, 2, 27, 24],
 [24, 5, 20, 37, 14],
 [2, 6, 38, 32],
 [20, 39, 44, 2, 42],
 [12, 2, 37, 44, 2],
 [37, 22, 40],
 [24, 39, 34, 2, 46],
 [14, 18, 39, 26, 43, 20],
 [43, 26, 39, 15],
 [43, 27, 39, 43, 24],
 [37, 29, 2, 31, 18],
 [43, 42, 39, 10, 2, 29],
 [9, 48, 6, 7, 20],
 [28, 10, 39, 30],
 [37, 30, 47, 45],
 [15, 28, 24, 32, 48, 46],
 [39, 39, 45, 2, 4],
 [43, 40, 49, 21],
 [37, 39, 18, 22, 

In [6]:
MAX_LENGTH = max(len(seq) for seq in encoded_reviews)
MAX_LENGTH

7

In [7]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded_reviews = pad_sequences(encoded_reviews, maxlen=MAX_LENGTH, padding="post")
print(len(padded_reviews[0]))

7


In [8]:
X = padded_reviews
y = sentiment
len(X), len(y)


(60, 60)

In [9]:
EMBEDDING_VECTOR_LENGTH = 10
tf.keras.backend.clear_session()

model = keras.Sequential(
    [
        keras.layers.Input(shape=(MAX_LENGTH,)),
        keras.layers.Embedding(
            input_dim=VOCAB_LENGTH,
            output_dim=EMBEDDING_VECTOR_LENGTH,
            name="embedding",
        ),
        keras.layers.GlobalAveragePooling1D(),
        keras.layers.Dense(1, activation=keras.activations.sigmoid),
    ]
)

2026-01-22 19:19:29.356311: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-22 19:19:29.356563: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-22 19:19:29.356568: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-22 19:19:29.357142: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-22 19:19:29.357158: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [10]:
model.compile(
    loss=keras.losses.binary_crossentropy,
    optimizer="adam",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 7, 10)          │           500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 10)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 511 (2.00 KB)

 Trainable params: 511 (2.00 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(X, y, epochs=5)

Epoch 1/5


2026-01-22 19:19:30.001998: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.4500 - loss: 0.6933
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4667 - loss: 0.6926
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5000 - loss: 0.6917
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5667 - loss: 0.6909
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6167 - loss: 0.6903


In [12]:
loss, accuracy = model.evaluate(X, y)
accuracy


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.6333 - loss: 0.6896 


0.6333333253860474

In [13]:
weights = model.get_layer("embedding").get_weights()[0]
weights.shape

(50, 10)

In [14]:
weights[0]

array([-0.04055661,  0.02670009,  0.02298402, -0.02352438,  0.03685192,
        0.02512075, -0.03723715, -0.04633338, -0.00358841, -0.0358394 ],
      dtype=float32)